In [ ]:
from pathlib import Path
import sys
_project_root = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from scipy.special import hermite
from math import factorial, pi, sqrt
import jax
import jax.numpy as jnp
import haiku as hk
from typing import Tuple
from src.core import Array


%config InlineBackend.figure_format = 'retina'

In [ ]:
# Define the normalized harmonic oscillator eigenfunction for quantum number n.
def phi(n, x):
	# Hermite polynomial of degree n
	Hn = hermite(n)
	# Normalization: (1/(2^n n! sqrt(pi)))^(1/2)
	norm = 1.0/np.sqrt(2**n * factorial(n)) * (1 / np.pi) ** 0.25
	return norm * np.exp(-x**2/2) * Hn(x)

def slater_wavefunction(x_vec):
	"""
	Slater determinant wave function for 3 fermions
	"""
	# x_vec is an array of positions [x1, x2, x3]
	# We use orbitals phi_0, phi_1, phi_2.
	# Build a 3x3 matrix: row n corresponds to orbital phi(n,x)
	M = np.array([phi(n, x_vec) for n in range(3)])
	# The determinant enforces antisymmetry (swapping any two x's flips the sign).
	return np.linalg.det(M)

In [ ]:
# Set up a grid for two coordinates, fixing the third.
x1_vals = np.linspace(-3, 3, 200)
x2_vals = np.linspace(-3, 3, 200)
X1, X2 = np.meshgrid(x1_vals, x2_vals)
fixed_x3 = 0.0  # Fix the coordinate of fermion 3

# Evaluate the wavefunction on the grid.
psi_vals = np.zeros_like(X1)
for i in range(X1.shape[0]):
	for j in range(X1.shape[1]):
		x_vec = np.array([X1[i, j], X2[i, j], fixed_x3])
		psi_vals[i, j] = slater_wavefunction(x_vec)

# Plot the wavefunction amplitude (which may be positive or negative)
fig, _axs = plt.subplots(1, 2, figsize=(12, 6), squeeze=False)
axs: list[Axes] = _axs.flatten().tolist()
contour = axs[0].contourf(X1, X2, psi_vals, levels=50, cmap='coolwarm')
#  color="#383b3e"
plt.colorbar(contour, label='Wavefunction amplitude')
# axs[0].set_facecolor("#f4f0e8")
axs[0].set(title=f'Slice of antisymmetric 3-fermion wavefunction (fixed $x_3={fixed_x3}$)', xlabel='$x_1$', ylabel='$x_2$', facecolor="#f4f0e8")
# Additionally, one may plot the probability density |Ψ|²:
density = psi_vals ** 2
contour2 = axs[1].contourf(X1, X2, density, levels=50, cmap='viridis')
plt.colorbar(contour2, label=r'$|\Psi|^2$')
axs[1].set(title=f'Probability density slice (fixed $x_3={fixed_x3}$)', xlabel='$x_1$', ylabel='$x_2$')
plt.show()



In [ ]:
def normalizing_flow(x: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
	"""
	Placeholder for CNF

	Returns
	----------------
	Assume returns transformed variables y and log Jacobian determinant
	"""
	y = x 
	log_det = jnp.zeros(x.shape[0])
	return y, log_det

def quantum_flow_forward(x: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
	"""
	
	Parameters
	----------------
	x : jnp.ndarray
	"""
	# Pushforward base samples
	y, log_det = normalizing_flow(x)
	
	# Compute amplitude. Ex: if base is a standard Gaussian:
	amplitude = jnp.exp(-0.5 * jnp.sum(y**2, axis=-1) - 0.5 * log_det)
	
	# Branch out to produce a phase. Network can be any MLP.
	phase_net = hk.nets.MLP([64, 64, 1])
	# pass base samples through phase network
	phase = phase_net(x)
	
	return amplitude, phase

def wavefunction(x: jnp.ndarray) -> jnp.ndarray:
	amplitude, phase = quantum_flow_forward(x)
	psi = amplitude * jnp.exp(1j * phase)  # complex wavefunction
	return psi

def forward_fn(x):
	return wavefunction(x)

forward = hk.transform(forward_fn)
x = jnp.zeros((10, 3))  # e.g., 3-dimensional input
params = forward.init(jax.random.PRNGKey(42), x)
psi_vals = forward.apply(params, None, x)
print(psi_vals)

### Density Evolution

In [ ]:
from physics.density_evolution import plot_density_matrix_heatmap

# Example usage:
N = 4  # dimension of your Hilbert space, for instance
rho_example = np.random.rand(N,N) + 1j*np.random.rand(N,N)
rho_example = 0.5 * (rho_example + rho_example.conj().T)  # make it Hermitian
rho_example = rho_example / np.trace(rho_example)        # trace-1, approximate density matrix

plot_density_matrix_heatmap(rho_example, title="Example Density Matrix (Real Part)")

In [ ]:
from matplotlib.animation import FuncAnimation

def animate_density_matrices(rho_list, interval=300):
  """
  Animate a sequence of density matrices (heatmaps). 
  rho_list is a list of 2D numpy arrays (complex) or times steps.
  interval is the delay between frames in milliseconds.
  """
  fig, ax = plt.subplots()
  cax = ax.matshow(rho_list[0].real, cmap='viridis')
  fig.colorbar(cax)
  ax.set_title("Time Evolution of Density Matrix (Real Part)")

  def update(frame):
    cax.set_array(rho_list[frame].real)
    ax.set_xlabel(f"Time step: {frame}")
    return [cax]

  anim = FuncAnimation(fig, update, frames=len(rho_list), interval=interval, blit=False)
  plt.show()

# Example usage:
# Suppose you have a function that computes rho(t).
N = 4
rho_0 = np.random.rand(N,N) + 1j*np.random.rand(N,N)
rho_0 = 0.5*(rho_0+rho_0.conj().T)  # Hermitian
rho_0 /= np.trace(rho_0)

# Fake time evolution:
ntimes = 20
rho_list = []
for t_idx in range(ntimes):
	# For demonstration: rotate phase by t_idx
	phase = np.exp(1j*0.1*t_idx)
	rho_t = rho_0 * phase
	rho_t = 0.5*(rho_t+rho_t.conj().T)
	rho_t /= np.trace(rho_t)
	rho_list.append(rho_t)

animate_density_matrices(rho_list, interval=300)